In [1]:
import requests, pandas as pd

In [9]:
CENSUS_KEY = "77e29a2ecf656c41fb214433c56d1f57dd6a6d0a"
year = 2024
variables = [
    "B19013_001E",  # median HH income
    "B01003_001E",  # total population
    "B03002_003E",  # White alone (non-Hispanic)
    "B03002_012E",  # Hispanic or Latino
    "B03002_004E",  # Black/AA
    "B25003_001E", "B25003_002E",  # housing tenure
    "B08301_001E", "B08301_002E",  # commute mode
    # Income brackets for OBBBA exposure intensity
    "B19001_001E","B19001_002E","B19001_003E","B19001_004E",
    "B19001_005E","B19001_006E","B19001_007E","B19001_008E",
    "B19001_009E","B19001_010E","B19001_011E","B19001_012E",
    "B19001_013E","B19001_014E","B19001_015E","B19001_016E","B19001_017E",
]
get_str = "NAME," + ",".join(variables)

In [10]:
url = f"https://api.census.gov/data/{year}/acs/acs5"
params = {
    "get": get_str,
    "for": "county:*",
    "in": "state:06",
    "key": CENSUS_KEY
}
r = requests.get(url, params=params)
acs = pd.DataFrame(r.json()[1:], columns=r.json()[0])

In [11]:
# Derive OBBBA subsidy exposure intensity
# $150k single / $300k joint threshold — proxy using HH income < $150k share
# B19001 brackets go up to bin 016 ($150k–$199k), bin 017 ($200k+)
income_cols = [f"B19001_{str(i).zfill(3)}E" for i in range(1, 18)]
acs[income_cols] = acs[income_cols].astype(float)
# Households earning under $150k = bins 002–015
under_150k = [f"B19001_{str(i).zfill(3)}E" for i in range(2, 16)]
acs["share_under_150k"] = acs[under_150k].sum(axis=1) / acs["B19001_001E"]

In [14]:
rename_map = {
    # ── Geography ────────────────────────────────────────────────────────────
    "NAME"          : "county_name",
    "state"         : "state_fips",
    "county"        : "county_fips",

    # ── Median household income ───────────────────────────────────────────────
    "B19013_001E"   : "median_hh_income",

    # ── Total population ──────────────────────────────────────────────────────
    "B01003_001E"   : "total_population",

    # ── Race / ethnicity ──────────────────────────────────────────────────────
    "B03002_003E"   : "pop_white_non_hispanic",
    "B03002_004E"   : "pop_black_non_hispanic",
    "B03002_012E"   : "pop_hispanic_any_race",

    # ── Housing tenure ────────────────────────────────────────────────────────
    "B25003_001E"   : "housing_units_total",
    "B25003_002E"   : "housing_units_owner_occupied",

    # ── Commute mode ──────────────────────────────────────────────────────────
    "B08301_001E"   : "commuters_total",
    "B08301_002E"   : "commuters_drove_alone",

    # ── Household income brackets (counts of households) ─────────────────────
    "B19001_001E"   : "hh_income_total",
    "B19001_002E"   : "hh_income_under_10k",
    "B19001_003E"   : "hh_income_10k_to_14999",
    "B19001_004E"   : "hh_income_15k_to_19999",
    "B19001_005E"   : "hh_income_20k_to_24999",
    "B19001_006E"   : "hh_income_25k_to_29999",
    "B19001_007E"   : "hh_income_30k_to_34999",
    "B19001_008E"   : "hh_income_35k_to_39999",
    "B19001_009E"   : "hh_income_40k_to_44999",
    "B19001_010E"   : "hh_income_45k_to_49999",
    "B19001_011E"   : "hh_income_50k_to_59999",
    "B19001_012E"   : "hh_income_60k_to_74999",
    "B19001_013E"   : "hh_income_75k_to_99999",
    "B19001_014E"   : "hh_income_100k_to_124999",
    "B19001_015E"   : "hh_income_125k_to_149999",
    "B19001_016E"   : "hh_income_150k_to_199999",
    "B19001_017E"   : "hh_income_200k_and_over",
}

acs = acs.rename(columns=rename_map)

In [15]:
# ── Sanity check: confirm no raw B-codes remain ───────────────────────────
leftover = [c for c in acs.columns if c.startswith("B")]
if leftover:
    print("Warning — unrenamed Census codes still present:", leftover)
else:
    print("All columns renamed successfully.")
    print(acs.columns.tolist())

All columns renamed successfully.
['county_name', 'median_hh_income', 'total_population', 'pop_white_non_hispanic', 'pop_hispanic_any_race', 'pop_black_non_hispanic', 'housing_units_total', 'housing_units_owner_occupied', 'commuters_total', 'commuters_drove_alone', 'hh_income_total', 'hh_income_under_10k', 'hh_income_10k_to_14999', 'hh_income_15k_to_19999', 'hh_income_20k_to_24999', 'hh_income_25k_to_29999', 'hh_income_30k_to_34999', 'hh_income_35k_to_39999', 'hh_income_40k_to_44999', 'hh_income_45k_to_49999', 'hh_income_50k_to_59999', 'hh_income_60k_to_74999', 'hh_income_75k_to_99999', 'hh_income_100k_to_124999', 'hh_income_125k_to_149999', 'hh_income_150k_to_199999', 'hh_income_200k_and_over', 'state_fips', 'county_fips', 'share_under_150k']


In [16]:
acs.head()

,county_name,median_hh_income,total_population,pop_white_non_hispanic,pop_hispanic_any_race,pop_black_non_hispanic,housing_units_total,housing_units_owner_occupied,commuters_total,commuters_drove_alone,...,hh_income_50k_to_59999,hh_income_60k_to_74999,hh_income_75k_to_99999,hh_income_100k_to_124999,hh_income_125k_to_149999,hh_income_150k_to_199999,hh_income_200k_and_over,state_fips,county_fips,share_under_150k
0,"Alameda County, California",129367,1649473,452099,386114,154136,598246,325509,837261,509744,...,23478.0,35075.0,56169.0,51989.0,44182.0,70712.0,192363.0,06,001,0.560256
1,"Alpine County, California",105521,1616,923,216,0,435,340,800,609,...,6.0,50.0,20.0,60.0,26.0,79.0,76.0,06,003,0.643678
2,"Amador County, California",88044,41428,30083,6634,790,16091,12941,15901,13086,...,1228.0,1293.0,2508.0,1639.0,1441.0,1962.0,1692.0,06,005,0.772917
3,"Butte County, California",67928,207929,135400,41660,3628,83312,47717,90412,73514,...,5463.0,7183.0,10137.0,7915.0,4973.0,6743.0,8681.0,06,007,0.814865
4,"Calaveras County, California",78647,46248,35337,6702,457,18437,15480,18264,14474,...,1335.0,1922.0,2178.0,1409.0,1783.0,1570.0,2516.0,06,009,0.778380


In [17]:
len(acs)

58

In [18]:
list(acs)

['county_name',
 'median_hh_income',
 'total_population',
 'pop_white_non_hispanic',
 'pop_hispanic_any_race',
 'pop_black_non_hispanic',
 'housing_units_total',
 'housing_units_owner_occupied',
 'commuters_total',
 'commuters_drove_alone',
 'hh_income_total',
 'hh_income_under_10k',
 'hh_income_10k_to_14999',
 'hh_income_15k_to_19999',
 'hh_income_20k_to_24999',
 'hh_income_25k_to_29999',
 'hh_income_30k_to_34999',
 'hh_income_35k_to_39999',
 'hh_income_40k_to_44999',
 'hh_income_45k_to_49999',
 'hh_income_50k_to_59999',
 'hh_income_60k_to_74999',
 'hh_income_75k_to_99999',
 'hh_income_100k_to_124999',
 'hh_income_125k_to_149999',
 'hh_income_150k_to_199999',
 'hh_income_200k_and_over',
 'state_fips',
 'county_fips',
 'share_under_150k']

In [19]:
acs.to_csv("census_acs_county_data.csv", index=False)